# 2.7 RAG and agentic search: chunking, embeddings, BM25

Step 2, lesson 7 of [Building with the Claude API](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287763) (Anthropic Academy).
RAG isn't an exam domain of its own, but it sits on top of three of them: **5 Context Management & Reliability** (what goes into the context window, and keeping track of where facts came from), **4 Prompt Engineering** (grounding prompts) and **2 Tool Design** (a search tool for an agent).

What this notebook covers, in the course's order:

1. Why RAG: documents too big, or too many, for the prompt
2. Chunking strategies and their trade-offs
3. Embeddings and cosine similarity, with a local model
4. A small in-memory vector index
5. BM25 lexical search, written from scratch
6. Hybrid search: combining both with reciprocal rank fusion
7. Answering a question with Claude from the retrieved chunks
8. Agentic search: letting Claude call a search tool as many times as it needs

> **Model used here:** `claude-sonnet-5-5`. Everything up to section 7 runs locally and costs nothing; sections 7 and 8 make 3–5 API requests in total.

## Setup

The same helpers as in [2.1](01-api-basics.ipynb). `OUT` is the step's `outputs/` folder (gitignored), where the embedding model is cached.

In [1]:
import math
import re
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
from anthropic import Anthropic
from dotenv import load_dotenv

warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm's notice about Jupyter widgets

load_dotenv()
client = Anthropic()
model = "claude-sonnet-5-5"
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)


def text_of(message):
    """Return the text of the first text block in a response."""
    return next(block.text for block in message.content if block.type == "text")


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})


def chat(messages, system=None, max_tokens=1000, **kwargs):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages, **kwargs}
    if system:
        params["system"] = system
    return text_of(client.messages.create(**params))

## 1. Why RAG

Say a user asks a question about a 300-page manual. Two ways to give Claude the facts:

| Option | Good | Bad |
|---|---|---|
| **Put the whole document in the prompt** | Simple; Claude sees everything | Every request pays for every token; it may not fit the context window at all (or a library of documents won't); long contexts dilute attention on the part that matters |
| **Retrieval-augmented generation (RAG)**: split the documents into chunks, find the few chunks relevant to *this* question, and put only those in the prompt | Cheap per request; scales to many documents | More moving parts; if retrieval misses the right chunk, Claude can't answer, so retrieval quality now matters |

A RAG pipeline has two phases:

- **Ahead of time (indexing):** split documents into chunks → turn each chunk into something searchable (an embedding vector, a keyword index, or both) → store it.
- **At question time:** search the index with the user's question → take the top few chunks → put them in the prompt with the question.

As the document, this notebook uses a short fictional employee handbook, written for this notebook, with Markdown headings. It's small enough to fit in a prompt; that keeps the demo cheap, but every step works the same with thousands of pages.

In [2]:
HANDBOOK = """# Nimbus Robotics employee handbook

## About this handbook
This handbook explains how we work at Nimbus Robotics, a 140-person company that builds warehouse robots. It applies to every employee in the Madrid and Lisbon offices. If a rule here conflicts with your contract, the contract wins. The People team updates the handbook every January.

## Working hours
Core hours are 10:00 to 16:00, Monday to Thursday, and 10:00 to 14:00 on Friday. Outside core hours you choose when to work, as long as you complete 38 hours a week. Overtime must be agreed with your manager in advance and is paid at 1.5 times the normal rate, or taken as time off within three months.

## Remote work
You can work from home up to three days a week. Your team agrees a shared office day so that people meet in person. Employees may also work from outside Spain and Portugal for up to 20 working days per calendar year, but only after passing probation. Requests for this go to the People team at least two weeks before the trip, because tax and insurance rules depend on the destination.

## Probation
New employees have a probation period of six months. During probation either side can end the contract with 15 days' notice. Your manager holds a check-in at the end of month three and a final review in month six.

## Holidays and leave
Everyone gets 25 days of paid holiday a year plus local public holidays. Up to five unused days can be carried over to the next year, and they expire on 31 March. Sick leave needs a doctor's note from the fourth day. Parents get 16 weeks of paid parental leave, which can be split into blocks of at least two weeks.

## Expenses and travel
Book travel through the TravelDesk portal and charge it to cost centre CC-7731 for customer visits or CC-1020 for internal trips. Train is preferred for trips under four hours. Hotels are capped at 150 euros a night, or 200 euros in London, Paris and Munich. Submit receipts within 30 days; claims older than 90 days are not reimbursed. Meals on business trips are covered up to 45 euros a day.

## Equipment and IT support
You get a laptop on your first day and a second monitor on request. Report problems through the IT portal or on the #it-help channel. If the VPN client shows error NX-4012, your device certificate has expired: open the IT portal and choose "Renew certificate". Error NX-2210 means your password expired and must be reset at the self-service page.

## Security
Lock your screen whenever you leave your desk. Never share passwords, even with IT staff. Customer data stays in approved systems and must not be copied to personal devices or personal cloud storage. Report a lost laptop or phone to security@nimbus.example within one hour.

## Performance reviews
Reviews happen twice a year, in June and December. You write a self-assessment, your manager adds feedback from two peers you choose, and you agree goals for the next half. Salary changes are decided once a year after the December review.

## Learning budget
Each employee has 1,000 euros a year for courses, books and conferences, and five learning days. Certification exams count towards the budget. Ask your manager before booking anything over 300 euros. The budget does not roll over.

## Leaving the company
The notice period is one month, or two months for managers. Return your laptop, badge and any other equipment on your last day. Unused holiday is paid out in your final salary.
"""
print(len(HANDBOOK), "characters,", len(HANDBOOK.split()), "words")

3432 characters, 600 words


## 2. Chunking strategies

A chunk should be small enough that a search hit is *about one thing*, and large enough to make sense on its own. The course compares three ways to cut:

| Strategy | How | Trade-off |
|---|---|---|
| **By size** | Fixed number of characters (or tokens), with some **overlap** between neighbours | Works on any text. Cuts through sentences and topics; the overlap softens that but duplicates text |
| **By sentence** | Group N sentences per chunk, overlapping by a sentence or two | Never cuts mid-sentence. Chunk sizes vary, and a topic can still be split |
| **By structure** | Split at headings, sections, functions, ... | Chunks match the author's own topics. Needs a document with reliable structure; sections can be very uneven in length |

A fourth option, **semantic chunking**, starts a new chunk where the meaning shifts (for example, where the embedding of consecutive sentences changes a lot). It is the most adaptive and the most expensive to compute.

In [3]:
def chunk_by_char(text, size=400, overlap=80):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks


def chunk_by_sentence(text, max_sentences=4, overlap=1):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, step = [], max_sentences - overlap
    for i in range(0, len(sentences), step):
        chunks.append(" ".join(sentences[i:i + max_sentences]))
        if i + max_sentences >= len(sentences):
            break
    return chunks


def chunk_by_section(text):
    parts = re.split(r"\n(?=## )", text.strip())       # split before every '## ' heading
    return [part.strip() for part in parts if part.startswith("## ")]


for name, chunks in [("by size", chunk_by_char(HANDBOOK)),
                     ("by sentence", chunk_by_sentence(HANDBOOK)),
                     ("by section", chunk_by_section(HANDBOOK))]:
    print(f"{name:12} {len(chunks):3} chunks, {min(map(len, chunks))}-{max(map(len, chunks))} characters")

by size       11 chunks, 232-400 characters
by sentence   14 chunks, 116-407 characters
by section    11 chunks, 199-417 characters


Look at how a size-based chunk starts and ends: it cuts through words and mixes two sections. The section-based chunk is one complete topic with its heading, which is also a useful label for citing the source later.

In [4]:
print("--- size-based chunk 3 ---")
print(repr(chunk_by_char(HANDBOOK)[3]))
print("\n--- section-based chunk 2 ---")
print(chunk_by_section(HANDBOOK)[2])

--- size-based chunk 3 ---
" People team at least two weeks before the trip, because tax and insurance rules depend on the destination.\n\n## Probation\nNew employees have a probation period of six months. During probation either side can end the contract with 15 days' notice. Your manager holds a check-in at the end of month three and a final review in month six.\n\n## Holidays and leave\nEveryone gets 25 days of paid holiday a y"

--- section-based chunk 2 ---
## Remote work
You can work from home up to three days a week. Your team agrees a shared office day so that people meet in person. Employees may also work from outside Spain and Portugal for up to 20 working days per calendar year, but only after passing probation. Requests for this go to the People team at least two weeks before the trip, because tax and insurance rules depend on the destination.


This handbook has clean headings, so we use **section chunks** from here on.

In [5]:
chunks = chunk_by_section(HANDBOOK)
title = lambda chunk: chunk.splitlines()[0].removeprefix("## ")   # the heading, used as a label
print([title(c) for c in chunks])

['About this handbook', 'Working hours', 'Remote work', 'Probation', 'Holidays and leave', 'Expenses and travel', 'Equipment and IT support', 'Security', 'Performance reviews', 'Learning budget', 'Leaving the company']


## 3. Embeddings and cosine similarity

An **embedding model** turns a piece of text into a vector of numbers (here 384 of them) such that texts with similar *meaning* get vectors pointing in similar directions. "My laptop was stolen" and "I lost my computer" share almost no words but end up close together.

**Cosine similarity** measures that closeness: the cosine of the angle between two vectors, from -1 (opposite) to 1 (same direction). If the vectors are normalised to length 1, it is just the dot product.

> **Changed since the course.** The course uses **Voyage AI** embeddings (Anthropic doesn't make an embedding model; its docs point to Voyage), which needs a separate Voyage API key. This notebook uses a small open-source model, **`BAAI/bge-small-en-v1.5`**, run locally on the CPU with [`fastembed`](https://github.com/qdrant/fastembed). No key, no cost; the model (about 65 MB) is downloaded once into `outputs/fastembed_cache/`. The ideas are the same; a larger model would simply retrieve better.

In [6]:
from fastembed import TextEmbedding

embedder = TextEmbedding("BAAI/bge-small-en-v1.5", cache_dir=str(OUT / "fastembed_cache"))

sentences = ["My laptop was stolen on the train.",
             "I lost my computer while travelling.",
             "Hotels are capped at 150 euros a night."]
vectors = np.array(list(embedder.embed(sentences)))
print("shape:", vectors.shape, "| length of each vector:", np.linalg.norm(vectors, axis=1).round(3))


def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


print("stolen laptop vs lost computer:", round(cosine(vectors[0], vectors[1]), 3))
print("stolen laptop vs hotel cap:    ", round(cosine(vectors[0], vectors[2]), 3))

shape: (3, 384) | length of each vector: [1. 1. 1.]
stolen laptop vs lost computer: 0.858
stolen laptop vs hotel cap:     0.437


## 4. A small vector index

A **vector index** stores one embedding per chunk and, given a query, returns the chunks whose embeddings are most similar. Production systems use a vector database (pgvector, Qdrant, Pinecone, ...) with approximate nearest-neighbour search; for 11 chunks a NumPy matrix product is enough.

One detail of BGE models: queries and passages are embedded slightly differently (`query_embed` adds a search instruction to the query), which is why the class uses two different methods.

In [7]:
class VectorIndex:
    def __init__(self, embedder):
        self.embedder, self.docs, self.vectors = embedder, [], None

    def add_documents(self, docs):
        self.docs = list(docs)
        self.vectors = np.array(list(self.embedder.passage_embed(self.docs)))  # normalised vectors

    def search(self, query, k=3):
        q = next(iter(self.embedder.query_embed(query)))
        sims = self.vectors @ q                       # cosine similarity, since all vectors have length 1
        best = np.argsort(-sims)[:k]
        return [(float(sims[i]), self.docs[i]) for i in best]


vector_index = VectorIndex(embedder)
vector_index.add_documents(chunks)

for score, chunk in vector_index.search("my computer got stolen"):
    print(f"{score:.3f}  {title(chunk)}")

0.645  Security
0.590  Equipment and IT support
0.528  Leaving the company


The query shares no words with the Security section ("Report a lost laptop or phone..."), yet it comes first. That is what embeddings are good at.

## 5. BM25 lexical search

Embeddings blur exact strings. A product code, an error number or a cost-centre id carries no "meaning" for the model, so a query like `CC-1020` may land on whichever section *looks* similar. **Lexical search** matches the actual words, and **BM25** is the standard way to score it. For each query term it adds up:

- **IDF** (inverse document frequency): rare terms count more. `cc-1020` appears in one chunk, `the` in all of them.
- **TF with saturation**: a term appearing 3 times helps more than once, but not 3 times more (`k1` controls how fast it saturates).
- **Length normalisation**: a hit in a short chunk counts more than in a long one (`b` controls how much).

$$\text{score}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\,(1 - b + b\,\frac{|d|}{\text{avgdl}})}, \qquad \text{IDF}(t) = \ln\!\left(\frac{N - n_t + 0.5}{n_t + 0.5} + 1\right)$$

It needs a tokenizer; this one lowercases, keeps codes like `nx-4012` as one token and drops a few very common words (without that, a word like "does" that happens to appear in only one chunk gets a high IDF and wins).

In [8]:
STOPWORDS = set(
    "a an and are as at be by can do does for from has have how i if in is it its my of on or so "
    "that the this to was what when where which who will with you your".split()
)


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower()) if t not in STOPWORDS]


class BM25Index:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs, self.term_counts, self.doc_freq = [], [], Counter()

    def add_documents(self, docs):
        for doc in docs:
            tokens = tokenize(doc)
            self.docs.append(doc)
            self.term_counts.append((Counter(tokens), len(tokens)))
            self.doc_freq.update(set(tokens))
        self.avgdl = sum(length for _, length in self.term_counts) / len(self.docs)

    def idf(self, term):
        n, df = len(self.docs), self.doc_freq.get(term, 0)
        return math.log((n - df + 0.5) / (df + 0.5) + 1)

    def score(self, query_terms, i):
        counts, length = self.term_counts[i]
        total = 0.0
        for term in query_terms:
            f = counts.get(term, 0)
            if f:
                norm = 1 - self.b + self.b * length / self.avgdl
                total += self.idf(term) * f * (self.k1 + 1) / (f + self.k1 * norm)
        return total

    def search(self, query, k=3):
        terms = tokenize(query)
        scored = [(self.score(terms, i), doc) for i, doc in enumerate(self.docs)]
        return sorted((s for s in scored if s[0] > 0), key=lambda s: -s[0])[:k]   # only chunks that match


bm25_index = BM25Index()
bm25_index.add_documents(chunks)


def compare(query, k=3):
    print("query:", query)
    print("  vector:", [(title(c), round(s, 2)) for s, c in vector_index.search(query, k)])
    print("  BM25:  ", [(title(c), round(s, 2)) for s, c in bm25_index.search(query, k)])


compare("CC-1020")
compare("my computer got stolen")

query: CC-1020
  vector: [('Equipment and IT support', 0.51), ('Leaving the company', 0.49), ('Learning budget', 0.49)]
  BM25:   [('Expenses and travel', 1.74)]
query: my computer got stolen
  vector: [('Security', 0.64), ('Equipment and IT support', 0.59), ('Leaving the company', 0.53)]
  BM25:   []


Each method fails where the other succeeds:

- For the bare code `CC-1020`, the vector index puts the wrong section first; BM25 finds the one chunk that contains the code.
- For "my computer got stolen", BM25 finds nothing at all (none of those words are in the handbook), while the vector index finds the Security section.

## 6. Hybrid search with reciprocal rank fusion

The two scores are on different scales (cosine ≈ 0–1, BM25 ≈ 0–10), so you can't simply add them. **Reciprocal rank fusion (RRF)** ignores the scores and uses only each chunk's **rank** in each list:

$$\text{RRF}(d) = \sum_{\text{lists}} \frac{1}{k + \text{rank}(d)}$$

`k = 60` is the usual constant: it keeps the first places from dominating too much. A chunk ranked well by both methods rises to the top; a chunk found by only one still gets in.

In [9]:
def reciprocal_rank_fusion(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] = scores.get(doc, 0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda item: -item[1])


class HybridRetriever:
    def __init__(self, *indexes):
        self.indexes = indexes

    def search(self, query, k=3, candidates=5):
        rankings = [[doc for _, doc in index.search(query, candidates)] for index in self.indexes]
        return reciprocal_rank_fusion(rankings)[:k]


retriever = HybridRetriever(vector_index, bm25_index)
for query in ["CC-1020", "my computer got stolen"]:
    print(query, "->", [(title(c), round(s, 4)) for c, s in retriever.search(query)])

CC-1020 -> [('Expenses and travel', 0.032), ('Equipment and IT support', 0.0164), ('Leaving the company', 0.0161)]
my computer got stolen -> [('Security', 0.0164), ('Equipment and IT support', 0.0161), ('Leaving the company', 0.0159)]


Both queries now get the right section first. The course builds the same thing as a "multi-index" retriever: any number of indexes, one fused ranking.

## 7. Answering with Claude

The last step is ordinary prompt engineering: put the retrieved chunks in the prompt, clearly separated from the question, and tell Claude to answer **only** from them and to say so when they don't contain the answer. XML tags keep the parts apart, and the section titles let Claude say where each fact came from.

In [10]:
def answer_with_rag(question, k=3):
    hits = retriever.search(question, k=k)
    context = "\n".join(
        f'<document title="{title(chunk)}">\n{chunk}\n</document>' for chunk, _ in hits
    )
    prompt = f"""Answer the employee's question using only the handbook excerpts below.
After each fact, name the section it comes from in square brackets.
If the excerpts don't contain the answer, say that the handbook excerpts don't cover it.

<excerpts>
{context}
</excerpts>

<question>{question}</question>"""
    print("retrieved:", [title(chunk) for chunk, _ in hits])
    return chat([{"role": "user", "content": prompt}], max_tokens=500, output_config={"effort": "low"})


print(answer_with_rag("What happens to my unused holiday days when I resign?"))

retrieved: ['Leaving the company', 'Holidays and leave', 'Remote work']


When you resign, your unused holiday is paid out in your final salary [Leaving the company].

The excerpts don't say how this is calculated, for example whether it covers carried-over days. They also don't say whether the carry-over rule applies when you leave. That rule allows up to five unused days to be carried into the next year, expiring on 31 March [Holidays and leave].


**Exam angle (Domains 4 and 5).** Two habits from this prompt come back in exam scenarios: (1) *grounding*: instruct Claude to answer only from the supplied material and to say when it isn't there, rather than hoping it won't fill gaps; (2) *provenance*: keep a source label attached to every chunk so the answer can say where each claim came from. When information from several sources is merged or summarised, losing the link between a claim and its source is a classic reliability failure.

## 8. Agentic search

One-shot RAG searches **once**, with the user's words. That fails on questions that need several facts which the question doesn't name. Here is one: the answer depends on the probation rule, but the question never mentions probation.

In [11]:
multi_hop = ("I started at Nimbus on 1 March 2027. Can I work from Brazil for three weeks in July 2027, "
             "and who do I ask?")
print([title(chunk) for chunk, _ in retriever.search(multi_hop)])

['Remote work', 'About this handbook', 'Holidays and leave']


The Probation section isn't in the top 3, so a one-shot pipeline would give Claude half the facts.

**Agentic search** turns retrieval into a **tool** and lets Claude call it as often as it needs, with its own queries. Claude can split the question into sub-questions (working abroad, probation) and search for each, or read one result, spot "only after passing probation" and search again. The path can differ from run to run; the printed trace shows the one it took. This is how Claude Code explores a codebase (with grep and file reads rather than a vector index).

The tool loop is the one from lessons 2.5–2.6: append the full assistant content, answer every `tool_use` with a `tool_result`, stop when `stop_reason` isn't `tool_use`, and cap the number of turns.

In [12]:
search_tool = {
    "name": "search_handbook",
    "description": (
        "Search the Nimbus Robotics employee handbook and return the 2 most relevant sections. "
        "Use short, specific queries about one topic (e.g. 'probation length', 'working abroad approval'). "
        "If a result mentions a rule that depends on another policy, search for that policy too."
    ),
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "What to look for in the handbook"}},
        "required": ["query"],
        "additionalProperties": False,
    },
}


def run_search_tool(query):
    hits = retriever.search(query, k=2)
    return "\n\n".join(chunk for chunk, _ in hits)


def agentic_search(question, max_turns=5):
    messages = [{"role": "user", "content": question}]
    system = "Answer questions about the employee handbook. Use the search tool; answer only from what it returns."
    for turn in range(max_turns):
        response = client.messages.create(
            model=model, max_tokens=1000, system=system, tools=[search_tool], messages=messages,
        )
        add_assistant_message(messages, response.content)          # full content, tool_use blocks included
        if response.stop_reason != "tool_use":
            return text_of(response)
        results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"turn {turn + 1}: search_handbook({block.input['query']!r})")
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                 "content": run_search_tool(block.input["query"])})
        add_user_message(messages, results)                         # all results in one user message
    return "Stopped: too many turns."


print("\n" + agentic_search(multi_hop))

turn 1: search_handbook('working abroad approval')
turn 1: search_handbook('probation length')



Probably not. A July trip would fall inside your probation period.

- **Probation:** The handbook sets probation at six months. Yours started on 1 March 2027, so it runs until about the end of August 2027.
- **Working abroad:** You can work from outside Spain and Portugal for up to 20 working days per calendar year, but only after you pass probation. Three weeks is about 15 working days, so the length is within the limit. The timing is the problem, because July is before probation ends.
- **Who to ask:** Requests go to the People team at least two weeks before the trip. They check tax and insurance rules for the destination.

The handbook doesn't mention exceptions. You could still ask the People team whether one is possible, but I can't promise it would be. Otherwise, you could move the trip to September or later. Your 20 days would still be available then, since the allowance is per calendar year.


The trade-off: agentic search finds facts a single query misses, but it costs a request per step and its path varies from run to run. One-shot retrieval is cheaper, faster and predictable. Use the simplest that answers your users' real questions.

> **Beyond this notebook.** Two refinements worth knowing by name:
> - **Reranking**: retrieve more candidates than you need (say 20), then let a model (Claude, or a dedicated reranker) order them by relevance to the question and keep the best few.
> - **Contextual retrieval** ([Anthropic's write-up](https://www.anthropic.com/news/contextual-retrieval)): before indexing, ask Claude to write one or two sentences placing each chunk in its document ("This section is from Nimbus's handbook and covers ...") and prepend them to the chunk, for both the embedding and the BM25 index. Chunks stop being ambiguous out of context. Prompt caching (lesson 2.8) makes it cheap, because the whole document is the same prefix for every chunk.

**Exam angle (Domains 1 and 2).** Agentic search is an agent loop: the model decides the next query. Its search tool's description says *when* to call it and how to write a query; a vague description ("searches stuff") is the first thing to fix when an agent searches badly, before adding a routing layer or another model. And the loop has a hard stop (`max_turns`) in code.

## Recap

- RAG = index chunks ahead of time, retrieve the relevant few per question, ground the answer in them.
- Chunking: by size (any text, cuts topics), by sentence, by structure (best when headings are reliable), semantic (adaptive, costly).
- Embeddings match meaning; BM25 matches exact terms such as codes and ids. Use both and fuse the rankings with RRF.
- Grounded prompt: excerpts in tags with source labels, "answer only from these", and an explicit way to say "not covered".
- Agentic search lets Claude search repeatedly, for multi-step questions, at the cost of more requests.

Next: [2.8 Features of Claude](08-features-of-claude.ipynb).